# 02 — Build KNN Model

Notebook 3/4 of the Content-Based pipeline.

**Configurable Parameters**
| Parameter | Description |
|-----------|-------------|
| `VERSION` | Data version to process (e.g., 'v2') |

**Output:** `cb_knn_model_{VERSION}.pkl`, `cb_item_index_mapping_{VERSION}.parquet`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pyarrow.parquet as pq
import pickle, gc, os, psutil
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

def print_ram(label=''):
    mem = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    tag = f' [{label}]' if label else ''
    print(f'  [RAM{tag}] {mem:.2f} GB')

# ============================================================
# CONFIG
# ============================================================
VERSION = 'v5'  # Change this to switch versions

BASE_DIR   = '/content/drive/My Drive/Thesis/book_recsys'
DATA_DIR   = os.path.join(BASE_DIR, 'data/processed')

BOOKS_PQ   = os.path.join(DATA_DIR, f'cb_books_{VERSION}.parquet')
SOUP_PQ    = os.path.join(DATA_DIR, f'cb_soup_texts_{VERSION}.parquet')
TFIDF_MAT  = os.path.join(DATA_DIR, f'cb_tfidf_item_vector_{VERSION}.npz')
KNN_OUT    = os.path.join(DATA_DIR, f'cb_knn_model_{VERSION}.pkl')
MAPPING_OUT = os.path.join(DATA_DIR, f'cb_item_index_mapping_{VERSION}.parquet')

print(f'Config done for VERSION: {VERSION}')
print_ram()

Config done for VERSION: v5
  [RAM] 0.25 GB


## 1. Load TF-IDF Sparse Matrix

In [3]:
print('Loading TF-IDF matrix...')
tfidf_matrix = sp.load_npz(TFIDF_MAT)
print(f'  Shape: {tfidf_matrix.shape}')
print_ram('after load matrix')

Loading TF-IDF matrix...
  Shape: (7227, 20000)
  [RAM [after load matrix]] 0.26 GB


## 2. Fit KNN Model

In [4]:
if os.path.exists(KNN_OUT):
    print(f'SKIP: {os.path.basename(KNN_OUT)} already exists.')
    with open(KNN_OUT, 'rb') as f:
        knn = pickle.load(f)
else:
    print('Fitting KNN...')
    knn = NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1)
    knn.fit(tfidf_matrix)
    with open(KNN_OUT, 'wb') as f:
        pickle.dump(knn, f, protocol=4)
    print(f'  Saved: {os.path.basename(KNN_OUT)}')
print_ram('after KNN')

Fitting KNN...
  Saved: cb_knn_model_v5.pkl
  [RAM [after KNN]] 0.26 GB


## 3. Build Item Index Mapping

In [5]:
if os.path.exists(MAPPING_OUT):
    print(f'SKIP: {os.path.basename(MAPPING_OUT)} already exists.')
    df_mapping = pd.read_parquet(MAPPING_OUT)
else:
    print(f'Building index mapping from {os.path.basename(SOUP_PQ)}...')
    book_ids = pq.read_table(SOUP_PQ, columns=['book_id']).column('book_id').to_pylist()
    df_mapping = pd.DataFrame({'row_index': range(len(book_ids)), 'book_id': book_ids})
    df_mapping.to_parquet(MAPPING_OUT, index=False)
    print(f'  Saved: {os.path.basename(MAPPING_OUT)}')
    del book_ids
    gc.collect()

book_id_to_idx = dict(zip(df_mapping['book_id'], df_mapping['row_index']))
idx_to_book_id = dict(zip(df_mapping['row_index'], df_mapping['book_id']))
print_ram('after mapping')

Building index mapping from cb_soup_texts_v5.parquet...
  Saved: cb_item_index_mapping_v5.parquet
  [RAM [after mapping]] 0.28 GB


## 4. Demo Inference — Top 10 Similar Books

In [6]:
df_books = pd.read_parquet(BOOKS_PQ, columns=['book_id', 'title', 'author', 'genres', 'average_rating'])
def get_recommendations(book_id, n=10):
    idx = book_id_to_idx.get(str(book_id))
    if idx is None: return None
    query_vec = tfidf_matrix[idx]
    distances, indices = knn.kneighbors(query_vec, n_neighbors=n + 1)
    rec_idx  = indices[0][1:]
    rec_sims = 1 - distances[0][1:]
    rec_book_ids = [idx_to_book_id[i] for i in rec_idx]
    result = df_books[df_books['book_id'].isin(rec_book_ids)].copy()
    id_order = {bid: rank for rank, bid in enumerate(rec_book_ids)}
    result['_rank'] = result['book_id'].map(id_order)
    result['similarity'] = result['book_id'].map(dict(zip(rec_book_ids, np.round(rec_sims, 4))))
    return result.sort_values('_rank').drop(columns=['_rank']).reset_index(drop=True)

DEMO_QUERIES = ['1', '5333265']
for bid in DEMO_QUERIES:
    meta = df_books[df_books['book_id'] == bid]
    if not meta.empty:
        print(f'\nQuery: {meta.iloc[0]["title"]}')
        recs = get_recommendations(bid, n=5)
        if recs is not None:
            print(recs[['title', 'similarity']].to_string(index=False))


Query: Harry Potter and the Half-Blood Prince (Harry Potter, #6)
                                                      title  similarity
     Harry Potter and the Deathly Hallows (Harry Potter #7)      0.5467
    Harry Potter and the Deathly Hallows (Harry Potter, #7)      0.5429
Harry Potter and the Philosopher's Stone (Harry Potter, #1)      0.4587
    Harry Potter and the Deathly Hallows (Harry Potter, #7)      0.4359
 Harry Potter and the Chamber of Secrets (Harry Potter, #2)      0.4007
